
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning" style="width: 600px">
</div>

# Demo 2.2: Building a Bronze-to-Silver Pipeline
## Overview
In this demo, we build a two-layer incremental pipeline: a **Bronze Streaming Table** that ingests raw CSV files via Auto Loader, and a **Silver Materialized View** that joins and enriches the data. We then configure **REFRESH POLICY** and **TRIGGER ON UPDATE** so that refreshing the Streaming Table automatically cascades to the Materialized View. No manual MV refresh needed.

Most pipeline objects are created in the **SQL Editor** (the natural tool for DDL). This notebook serves as the guide: it provides the SQL to copy, verification queries, and teaching callouts.

## Learning Objectives
By the end of this demo, you will be able to:
- Use **EXPLAIN CREATE MATERIALIZED VIEW** to verify incrementalizability before building
- Create a **Streaming Table** and **Materialized View** using the SQL Editor
- Configure **REFRESH POLICY INCREMENTAL** on a Materialized View
- Set **TRIGGER ON UPDATE** to automatically cascade refreshes from ST to MV
- Demonstrate **incremental processing** triggered by new data arriving in the pipeline

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select a Serverless SQL Warehouse</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Starter Warehouse** (included in Databricks Free Edition), or another serverless SQL warehouse you have access to
  - Select **Compute** > **More** > **SQL Warehouse** > Select a SQL Warehouse you have access to.

**NOTE:** This notebook was **developed and tested using a Serverless SQL Warehouse**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>

## A. REQUIRED - Classroom Setup

<div style="border-left: 4px solid #d97706; background: #fff; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#92400e; margin-bottom:6px; font-size: 1.1em;">Run the Setup Cell</strong>
  <div style="color:#333;">
    Run the following cell to configure your environment. This resets the Volume to a clean state, exports the course data as CSV files, and creates the <code>bronze</code> and <code>silver</code> schemas. It also pre-creates the other source streaming tables (<code>bronze.raw_products</code>, <code>bronze.customer_feedback</code>) so you can focus on building <code>bronze.raw_orders</code> and the silver MV in this demo.
  </div>
</div>

In [0]:
%run ../Includes/Classroom-Setup-2.2-Demo

Workspace,User Catalog Name,Catalog Status
Learner Workspace,learn_etl_pipelines_sql_joemar_brillo,Catalog Already Exists


message
The operation was successfully executed.


message
The operation was successfully executed.


Status,Catalog,Working Schema,Data Schema
Setup Complete,learn_etl_pipelines_sql_joemar_brillo,bronze,data



## B. The Pipeline We Are Building

This demo builds the **Bronze → Silver** segment of the course pipeline. Gold aggregations, AUTO CDC, and orchestration are covered in later demos.


<span style="color:#991b1b; font-weight:bold; font-style:italic;">If the diagram below does not render, click into the cell and then click out. This triggers the Mermaid library to load.</span>


<div style="max-width: 900px; margin: 0 auto;">
<div>
<div class="mermaid" id="diagram-2-2-pipeline-architecture" style="font-size: 1em;">
flowchart TD
    subgraph BRONZE["<b>BRONZE: Streaming Tables</b>"]
        RO["raw_orders<br/><small>ST · Auto Loader · CSV from Volumes</small>"]
        RP["raw_products<br/><small>ST · Auto Loader · CSV from Volumes</small>"]
    end
    RC["data.raw_customers<br/><small>Unity Catalog managed table · joined directly</small>"]
    subgraph SILVER["<b>SILVER: Materialized View</b>"]
        OE["orders_enriched<br/><small>MV · joins orders + products + customers</small>"]
    end
    subgraph LATER["<b>Built in Later Demos</b>"]
        direction LR
        CDC["AUTO CDC (Module 3)"]
        GOLD["Gold MVs (Module 4)"]
    end
    RO -->|"TRIGGER ON UPDATE"| OE
    RP --> OE
    RC --> OE
    OE -.-> GOLD
    classDef bronze fill:#FFF3E0,stroke:#e65100,stroke-width:2px,color:#333
    classDef silver fill:#E3F2FD,stroke:#1565c0,stroke-width:2px,color:#333
    classDef source fill:#F3E5F5,stroke:#7b1fa2,stroke-width:2px,color:#333
    classDef later fill:#F5F5F5,stroke:#9e9e9e,stroke-width:1px,stroke-dasharray:5 5,color:#999
    class RO,RP bronze
    class OE silver
    class RC source
    class CDC,GOLD later
    style BRONZE fill:#FFF8E1,stroke:#e65100,stroke-width:2px
    style SILVER fill:#E8EAF6,stroke:#1565c0,stroke-width:2px
    style LATER fill:#FAFAFA,stroke:#bdbdbd,stroke-width:1px,stroke-dasharray:5 5
</div>
</div>
</div>

<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs";
mermaid.initialize({ startOnLoad: false });
await new Promise(r => requestAnimationFrame(r));
try {
  await mermaid.run({ querySelector: "#diagram-2-2-pipeline-architecture" });
} catch(e) {
  await new Promise(r => setTimeout(r, 1000));
  await mermaid.run({ querySelector: "#diagram-2-2-pipeline-architecture" });
}
document.querySelectorAll('#diagram-2-2-pipeline-architecture svg text, #diagram-2-2-pipeline-architecture svg .nodeLabel, #diagram-2-2-pipeline-architecture svg foreignObject div, #diagram-2-2-pipeline-architecture svg span').forEach(el => { el.style.fontSize = '1em'; });
</script>

**What we build in this demo:**
- Two **Streaming Tables** (`bronze.raw_orders` and `bronze.raw_products`) that ingest raw CSV files incrementally via Auto Loader
- A **Materialized View** (`silver.orders_enriched`) that joins orders with products and customers
- **REFRESH POLICY INCREMENTAL** to enforce incremental refresh on the MV
- **TRIGGER ON UPDATE** so refreshing the ST automatically cascades to the MV

## C. EXPLAIN Before You Build

Before we create any pipeline objects, let's check whether the query we plan for the MV **can** be refreshed incrementally. `EXPLAIN CREATE MATERIALIZED VIEW` analyzes the query plan and reports whether incremental refresh is possible.

We use the `data` schema tables (regular UC managed tables pre-loaded by setup) to test our planned query. This lets us see what properties are needed and fix them before building the pipeline.

**Note on schemas:** The `data` schema contains the original UC managed tables loaded during setup. The `bronze` schema will hold the Streaming Tables we create later. EXPLAIN runs against `data` (to test the query before we build), while the actual MV will source from `bronze`.

### C1. Run EXPLAIN: BEFORE Setting Properties

Let's run `EXPLAIN CREATE MATERIALIZED VIEW` against the query we plan to use. EXPLAIN requires a view name in the syntax. We use a placeholder name since it does not actually create anything.

**📌 Note:** `EXPLAIN CREATE MATERIALIZED VIEW` is **generally available** on Serverless SQL Warehouses.

In [0]:
select my_catalog;

my_catalog
learn_etl_pipelines_sql_joemar_brillo


In [0]:
EXPLAIN CREATE MATERIALIZED VIEW IDENTIFIER(my_catalog || '.data._explain_check')
AS
SELECT
  o.order_id,
  o.customer_id,
  o.product_id,
  CAST(o.order_date AS DATE)                          AS order_date,
  CAST(o.quantity AS INT)                              AS quantity,
  CAST(o.unit_price AS DECIMAL(10,2))                  AS unit_price,
  CAST(o.quantity * o.unit_price AS DECIMAL(12,2))     AS line_total,
  COALESCE(o.discount_pct, 0)                          AS discount_pct,
  COALESCE(o.shipping_cost, 0.00)                      AS shipping_cost,
  o.order_status,
  o.payment_method,
  COALESCE(c.first_name || ' ' || c.last_name, 'Unknown') AS customer_name,
  c.customer_segment,
  c.city                                                AS customer_city,
  c.state                                               AS customer_state,
  COALESCE(p.product_name, 'Unknown Product')           AS product_name,
  p.category                                            AS product_category,
  p.brand                                               AS product_brand,
  CAST(p.list_price AS DECIMAL(10,2))                   AS product_list_price
FROM IDENTIFIER(my_catalog || '.data.raw_orders') o
LEFT JOIN IDENTIFIER(my_catalog || '.data.raw_customers') c
  ON o.customer_id = c.customer_id
LEFT JOIN IDENTIFIER(my_catalog || '.data.raw_products') p
  ON o.product_id = p.product_id;

plan


<div style="border-left: 4px solid #2e7d32; background: #f7fdf4; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#1b5e20; margin-bottom:6px; font-size: 1.1em;">What to observe</strong>
  <div style="color:#333;">
    <ul>
      <li>The EXPLAIN output shows the view is <strong>not incrementalizable</strong></li>
      <li>It reports error conditions such as <code>ROW_TRACKING_NOT_ENABLED</code>, <code>CDF_NOT_ENABLED</code>, and <code>DELETION_VECTORS_NOT_ENABLED</code>, listing the specific tables that need changes</li>
      <li>It tells you exactly what to fix. No guesswork needed.</li>
    </ul>
    <p style="margin-top:10px;"><strong>What to observe:</strong> Always run EXPLAIN before deploying a production MV. It is your pre-deployment diagnostic tool: it tells you exactly what needs to change to enable incremental refresh, saving you from full recomputation on every refresh.</p>
  </div>
</div>

### C2. Set Source Table Properties for Incremental Refresh

The EXPLAIN output told us that the source tables need specific Delta features enabled:
- **Change Data Feed**: tracks row-level changes
- **Row Tracking**: enables precise change identification
- **Deletion Vectors**: enables efficient deletes

For **regular UC managed tables** like these, we use `ALTER TABLE ... SET TBLPROPERTIES`. (For **Streaming Tables**, these properties must be set at creation time: they cannot be altered after the fact.)

In [0]:
ALTER TABLE IDENTIFIER(my_catalog || '.data.raw_orders')
SET TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.enableRowTracking' = 'true',
  'delta.enableDeletionVectors' = 'true'
);

In [0]:
ALTER TABLE IDENTIFIER(my_catalog || '.data.raw_customers')
SET TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.enableRowTracking' = 'true',
  'delta.enableDeletionVectors' = 'true'
);

In [0]:
ALTER TABLE IDENTIFIER(my_catalog || '.data.raw_products')
SET TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.enableRowTracking' = 'true',
  'delta.enableDeletionVectors' = 'true'
);

<div style="border-left: 4px solid #2e7d32; background: #f7fdf4; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#1b5e20; margin-bottom:6px; font-size: 1.1em;">What to observe</strong>
  <div style="color:#333;">
    Each ALTER TABLE should complete successfully. These properties enable Delta Lake to track changes at the row level, which is what allows Materialized Views to refresh incrementally rather than recomputing the full dataset.
  </div>
</div>

### C3. Re-Run EXPLAIN: AFTER Setting Properties

Now that the source table properties are set, let's run EXPLAIN again. This time, the output should confirm that the view **is incrementalizable**.

In [0]:
EXPLAIN CREATE MATERIALIZED VIEW IDENTIFIER(my_catalog || '.data._explain_check')
AS
SELECT
  o.order_id,
  o.customer_id,
  o.product_id,
  CAST(o.order_date AS DATE)                          AS order_date,
  CAST(o.quantity AS INT)                              AS quantity,
  CAST(o.unit_price AS DECIMAL(10,2))                  AS unit_price,
  CAST(o.quantity * o.unit_price AS DECIMAL(12,2))     AS line_total,
  COALESCE(o.discount_pct, 0)                          AS discount_pct,
  COALESCE(o.shipping_cost, 0.00)                      AS shipping_cost,
  o.order_status,
  o.payment_method,
  COALESCE(c.first_name || ' ' || c.last_name, 'Unknown') AS customer_name,
  c.customer_segment,
  c.city                                                AS customer_city,
  c.state                                               AS customer_state,
  COALESCE(p.product_name, 'Unknown Product')           AS product_name,
  p.category                                            AS product_category,
  p.brand                                               AS product_brand,
  CAST(p.list_price AS DECIMAL(10,2))                   AS product_list_price
FROM IDENTIFIER(my_catalog || '.data.raw_orders') o
LEFT JOIN IDENTIFIER(my_catalog || '.data.raw_customers') c
  ON o.customer_id = c.customer_id
LEFT JOIN IDENTIFIER(my_catalog || '.data.raw_products') p
  ON o.product_id = p.product_id;

plan


<div style="border-left: 4px solid #2e7d32; background: #f7fdf4; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#1b5e20; margin-bottom:6px; font-size: 1.1em;">What to observe</strong>
  <div style="color:#333;">
    <ul>
      <li>The EXPLAIN output should now report that the view <strong>is incrementalizable</strong></li>
      <li>Compare this to the previous EXPLAIN output: the difference is entirely due to the table properties we just set</li>
    </ul>
    <p style="margin-top:10px;"><strong>What to observe:</strong> This is the "before and after" moment. The same query, the same tables. Now the engine knows it can process only changed rows instead of recomputing everything. We are confident our pipeline will support incremental refresh, so let's build it.</p>
  </div>
</div>

## D. Create the Bronze Streaming Table

Our source data lives as CSV files in a Volume directory (exported by the setup script from the `data.raw_orders` UC managed table). We will create a **Streaming Table** that ingests these files incrementally using `read_files()` with Auto Loader.

Streaming Tables are ideal for append-only ingestion patterns: they track which files have been processed and only pick up new ones on each refresh.

### D1. Preview the Source Data

Before building the Streaming Table, let's preview the raw orders data to understand its structure.

In [0]:
SELECT *
FROM IDENTIFIER(my_catalog || '.data.raw_orders')
LIMIT 10;

order_id,customer_id,product_id,order_date,quantity,unit_price,discount_pct,shipping_cost,order_status,payment_method,store_id,inserted_at
1,1992,54,2025-12-01T03:55:12.000Z,4,76.91,0.1127,19.16,completed,credit_card,2,2026-03-05T19:53:15.000Z
2,2288,45,2024-09-28T02:19:16.000Z,8,268.95,0.3,23.64,returned,apple_pay,7,2026-03-17T03:01:19.000Z
3,1098,48,2024-05-20T19:20:23.000Z,4,460.83,0.0656,8.59,pending,apple_pay,6,2026-03-26T15:39:02.000Z
4,2004,75,2026-02-11T13:22:55.000Z,1,23.82,0.0708,1.42,cancelled,apple_pay,3,2026-03-17T17:07:33.000Z
5,599,183,2025-07-29T09:55:12.000Z,5,312.75,0.1181,0.81,completed,credit_card,2,2026-03-14T18:47:09.000Z
6,1009,24,2026-01-02T19:26:11.000Z,2,284.87,0.0424,8.16,pending,apple_pay,2,2026-03-07T20:13:28.000Z
7,35,147,2026-03-01T10:50:06.000Z,3,451.83,0.1531,0.18,completed,apple_pay,5,2026-03-20T19:52:52.000Z
8,616,184,2024-05-04T21:09:07.000Z,3,497.56,0.0227,15.49,pending,apple_pay,1,2026-03-03T11:24:04.000Z
9,1344,177,2026-02-20T15:29:54.000Z,4,226.39,0.1898,2.33,returned,paypal,6,2026-03-13T13:43:44.000Z
10,1164,73,2025-07-26T06:36:43.000Z,3,250.88,0.0223,7.72,completed,apple_pay,10,2026-03-03T21:19:28.000Z


<div style="border-left: 4px solid #2e7d32; background: #f7fdf4; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#1b5e20; margin-bottom:6px; font-size: 1.1em;">What to observe</strong>
  <div style="color:#333;">
    <ul>
      <li>The <code>raw_orders</code> table contains order-level transaction data with fields like <code>order_id</code>, <code>customer_id</code>, <code>product_id</code>, <code>order_date</code>, <code>quantity</code>, <code>unit_price</code>, and <code>order_status</code></li>
      <li>Some fields are stored as strings that we will later cast to appropriate types in the Silver layer</li>
    </ul>
  </div>
</div>

**Your catalog name:** you will need this when creating objects in the SQL Editor:

In [0]:
SELECT my_catalog AS your_catalog_name;

your_catalog_name
learn_etl_pipelines_sql_joemar_brillo


### D2. Create the Streaming Table in the SQL Editor

We create the Streaming Table in the **SQL Editor** because it provides a clean environment for DDL: no `IDENTIFIER()` wrappers or `EXECUTE IMMEDIATE` needed.

Notice the `TBLPROPERTIES`: we enable Change Data Feed, Row Tracking, and Deletion Vectors at creation time because Streaming Table properties **cannot be altered after the fact**.

<div style="border-left: 4px solid #d97706; background: #fffbeb; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#92400e; margin-bottom:6px; font-size: 1.1em;">Copy to SQL Editor</strong>
  <div style="color:#333;">
    <ol>
      <li>Open the <strong>SQL Editor</strong> (click <strong>SQL Editor</strong> in the left sidebar)</li>
      <li><span style="background:#FEF3C7; padding:2px 6px; border-radius:3px;"><strong>Set the default catalog</strong> to your catalog (e.g.,<code>learn_etl_pipelines_sql_YOUR_USERNAME</code>) using the catalog picker at the top</span></li>
      <li>Copy the SQL below and run it:</li>
    </ol>
  </div>
</div>
<button onclick="copySTOrders()">Copy to clipboard</button>
<pre id="copy-block-st-orders" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#1B3139; color:#E8E3DC; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>DROP TABLE IF EXISTS bronze.raw_orders;

CREATE OR REFRESH STREAMING TABLE bronze.raw_orders
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.enableRowTracking' = 'true',
  'delta.enableDeletionVectors' = 'true'
)
AS SELECT * FROM STREAM read_files(
  '/Volumes/<mark style="background:#FEF3C7;color:#000;padding:1px 3px;">ADD_YOUR_CATALOG_HERE</mark>/data/course_data/raw_orders',
  format =&gt; 'csv',
  header =&gt; 'true',
  inferColumnTypes =&gt; 'true',
  schemaEvolutionMode =&gt; 'rescue'
);
</code></pre>
<script>
function copySTOrders() {
  var el = document.getElementById("copy-block-st-orders");
  if (!el) return;
  var text = el.innerText;
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text).then(function() { alert("Copied to clipboard"); }).catch(function() { fallbackCopyST(text); });
  } else { fallbackCopyST(text); }
}
function fallbackCopyST(text) {
  var textarea = document.createElement("textarea"); textarea.value = text; textarea.style.position = "fixed"; textarea.style.left = "-9999px";
  document.body.appendChild(textarea); textarea.select();
  try { document.execCommand("copy"); alert("Copied to clipboard"); } catch(e) { alert("Copy failed. Select and copy manually."); }
  finally { document.body.removeChild(textarea); }
}
</script>
<p style="margin-top:8px; color:#555;"><strong>Replace <code><mark style="background:#FEF3C7;color:#000;padding:1px 3px;">ADD_YOUR_CATALOG_HERE</mark></code> with your catalog name from the cell output above (e.g., <code>learn_etl_pipelines_sql_first_last</code>)</strong> with your actual catalog name. After the statement completes, return to this notebook and continue with D3.</p>

### D3. Create the Products Streaming Table in the SQL Editor

Now create a second Streaming Table for the product catalog. Same pattern. Copy to the SQL Editor.

<div style="border-left: 4px solid #d97706; background: #fffbeb; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#92400e; margin-bottom:6px; font-size: 1.1em;">Copy to SQL Editor</strong>
  <div style="color:#333;">
    <p>Paste and run in the same SQL Editor tab (your catalog should still be set):</p>
  </div>
</div>
<button onclick="copySTProducts()">Copy to clipboard</button>
<pre id="copy-block-st-products" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#1B3139; color:#E8E3DC; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>DROP TABLE IF EXISTS bronze.raw_products;

CREATE OR REFRESH STREAMING TABLE bronze.raw_products
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.enableRowTracking' = 'true',
  'delta.enableDeletionVectors' = 'true'
)
AS SELECT * FROM STREAM read_files(
  '/Volumes/<mark style="background:#FEF3C7;color:#000;padding:1px 3px;">ADD_YOUR_CATALOG_HERE</mark>/data/course_data/raw_products',
  format =&gt; 'csv',
  header =&gt; 'true',
  inferColumnTypes =&gt; 'true',
  schemaEvolutionMode =&gt; 'rescue'
);
</code></pre>
<script>
function copySTProducts() {
  var el = document.getElementById("copy-block-st-products");
  if (!el) return;
  var text = el.innerText;
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text).then(function() { alert("Copied to clipboard"); }).catch(function() { fallbackCopyProd(text); });
  } else { fallbackCopyProd(text); }
}
function fallbackCopyProd(text) {
  var textarea = document.createElement("textarea"); textarea.value = text; textarea.style.position = "fixed"; textarea.style.left = "-9999px";
  document.body.appendChild(textarea); textarea.select();
  try { document.execCommand("copy"); alert("Copied to clipboard"); } catch(e) { alert("Copy failed. Select and copy manually."); }
  finally { document.body.removeChild(textarea); }
}
</script>
<p style="margin-top:8px; color:#555;"><strong>Replace <code><mark style="background:#FEF3C7;color:#000;padding:1px 3px;">ADD_YOUR_CATALOG_HERE</mark></code> with your catalog name from the cell output above (e.g., <code>learn_etl_pipelines_sql_first_last</code>)</strong> with your catalog name. After the statement completes, return to this notebook and continue with D4.</p>

### D4. Verify the Bronze Tables

Let's confirm the row counts for both Streaming Tables.

In [0]:
SELECT 'bronze.raw_orders' AS table_name, COUNT(*) AS row_count
FROM IDENTIFIER(my_catalog || '.bronze.raw_orders')
UNION ALL
SELECT 'bronze.raw_products', COUNT(*)
FROM IDENTIFIER(my_catalog || '.bronze.raw_products');

table_name,row_count
bronze.raw_orders,15000
bronze.raw_products,200


<div style="border-left: 4px solid #2e7d32; background: #f7fdf4; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#1b5e20; margin-bottom:6px; font-size: 1.1em;">What to observe</strong>
  <div style="color:#333;">
    <ul>
      <li><strong>bronze.raw_orders</strong> should contain approximately <strong>15,000 rows</strong></li>
      <li><strong>bronze.raw_products</strong> should contain approximately <strong>200 rows</strong> (the product catalog)</li>
      <li>Both schemas were auto-inferred from CSV files by Auto Loader with <code>inferColumnTypes</code></li>
    </ul>
  </div>
</div>

## E. Create the Silver Materialized View

Now we build the Silver layer. The **Materialized View** joins orders with customers and products to create a denormalized, analysis-ready table. We apply type casting, null handling, and enrichment joins.

**Why CAST when `inferColumnTypes` is enabled?** This is *defensive typing*: Auto Loader infers types from the first batch of files, but CSV inference is best-effort. Explicit CASTs in the MV guarantee the Silver layer has the exact types downstream consumers expect, regardless of how the source schema was inferred.

**📌 Note:** This MV joins `data.raw_customers` directly (a regular UC managed table) as a simplification for this demo. In the full pipeline (Module 3+), this join will target the CDC-processed Silver customer table instead.

### E1. Create the MV in the SQL Editor

<div style="border-left: 4px solid #d97706; background: #fffbeb; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#92400e; margin-bottom:6px; font-size: 1.1em;">Copy to SQL Editor</strong>
  <div style="color:#333;">
    <p>In the <strong>SQL Editor</strong>:</p>
    <ol>
      <li>Confirm the <strong>default catalog</strong> is set to your <code>learn_etl_pipelines_sql_YOUR_USERNAME</code> catalog</li>
      <li>Confirm the <strong>default schema</strong> is set to <code>default</code></li>
      <li>Copy the SQL below and run it. Hover over each highlighted section to explore what it does.</li>
    </ol>
  </div>
</div>

<style>
.e1mv-rp { display:inline-block; width:100%; background:rgba(0,169,114,0.2); padding:0 4px; border-radius:3px; cursor:pointer; transition:background 0.2s; }
.e1mv-rp:hover { background:rgba(0,169,114,0.4); }
.e1mv-card { position:absolute; left:50%; top:80px; width:260px; background:#F9F7F4; border-radius:8px; border-left:5px solid #00A972; padding:12px 14px; font-size:0.75rem; line-height:1.5; color:#444; opacity:0; transition:opacity 0.3s; pointer-events:none; box-shadow:0 4px 16px rgba(0,0,0,0.3); font-family:system-ui,sans-serif; z-index:10; word-wrap:break-word; overflow-wrap:break-word; white-space:normal; }
.e1mv-rp:hover + .e1mv-card, .e1mv-card:hover { opacity:1; }
</style>
<button onclick="e1CopyMVSQL()">Copy to clipboard</button>
<div style="position:relative; display:inline-block; width:100%;">
<pre id="copy-block-mv" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#1B3139; color:#E8E3DC; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>DROP MATERIALIZED VIEW IF EXISTS silver.orders_enriched;

CREATE OR REFRESH MATERIALIZED VIEW silver.orders_enriched
<span class="e1mv-rp">REFRESH POLICY INCREMENTAL</span><span class="e1mv-card"><strong style="color:#00A972;font-size:0.9rem;">REFRESH POLICY INCREMENTAL</strong><br/><br/>Prefers <strong>incremental refresh</strong>: processing only changed rows when feasible. Falls back to full recompute if incremental becomes unavailable.<br/><br/><strong>Fails at CREATE</strong> if the query cannot be incrementalized. This is why we verified with EXPLAIN first (Section C).<br/><br/>Must be set at <strong>creation time</strong>: cannot be added via ALTER.</span>
AS
SELECT
  o.order_id,
  o.customer_id,
  o.product_id,
  CAST(o.order_date AS DATE)                          AS order_date,
  CAST(o.quantity AS INT)                              AS quantity,
  CAST(o.unit_price AS DECIMAL(10,2))                  AS unit_price,
  CAST(o.quantity * o.unit_price AS DECIMAL(12,2))     AS line_total,
  COALESCE(o.discount_pct, 0)                          AS discount_pct,
  COALESCE(o.shipping_cost, 0.00)                      AS shipping_cost,
  o.order_status,
  o.payment_method,
  COALESCE(c.first_name || ' ' || c.last_name, 'Unknown') AS customer_name,
  c.customer_segment,
  c.city                                                AS customer_city,
  c.state                                               AS customer_state,
  COALESCE(p.product_name, 'Unknown Product')           AS product_name,
  p.category                                            AS product_category,
  p.brand                                               AS product_brand,
  CAST(p.list_price AS DECIMAL(10,2))                   AS product_list_price
FROM bronze.raw_orders o
LEFT JOIN data.raw_customers c
  ON o.customer_id = c.customer_id
LEFT JOIN bronze.raw_products p
  ON o.product_id = p.product_id;
</code></pre>
</div>
<textarea id="e1mv-sql-text" style="position:absolute;left:-9999px;">DROP MATERIALIZED VIEW IF EXISTS silver.orders_enriched;
CREATE OR REFRESH MATERIALIZED VIEW silver.orders_enriched
REFRESH POLICY INCREMENTAL
AS
SELECT
  o.order_id,
  o.customer_id,
  o.product_id,
  CAST(o.order_date AS DATE)                          AS order_date,
  CAST(o.quantity AS INT)                              AS quantity,
  CAST(o.unit_price AS DECIMAL(10,2))                  AS unit_price,
  CAST(o.quantity * o.unit_price AS DECIMAL(12,2))     AS line_total,
  COALESCE(o.discount_pct, 0)                          AS discount_pct,
  COALESCE(o.shipping_cost, 0.00)                      AS shipping_cost,
  o.order_status,
  o.payment_method,
  COALESCE(c.first_name || ' ' || c.last_name, 'Unknown') AS customer_name,
  c.customer_segment,
  c.city                                                AS customer_city,
  c.state                                               AS customer_state,
  COALESCE(p.product_name, 'Unknown Product')           AS product_name,
  p.category                                            AS product_category,
  p.brand                                               AS product_brand,
  CAST(p.list_price AS DECIMAL(10,2))                   AS product_list_price
FROM bronze.raw_orders o
LEFT JOIN data.raw_customers c
  ON o.customer_id = c.customer_id
LEFT JOIN bronze.raw_products p
  ON o.product_id = p.product_id;</textarea>
<script>
function e1CopyMVSQL() {
  var el = document.getElementById('e1mv-sql-text');
  if (!el) return;
  el.style.position = 'fixed'; el.style.left = '0'; el.style.top = '0';
  el.select();
  try { document.execCommand('copy'); alert('Copied to clipboard'); } catch(e) { alert('Copy failed'); }
  el.style.position = 'absolute'; el.style.left = '-9999px';
}
</script>
<p style="margin-top:8px; color:#555;">After the statement completes in the SQL Editor, return to this notebook and continue with E2.</p>

### E2. Verify the Materialized View

Let's preview the enriched data and confirm the joins worked correctly.

In [0]:
SELECT *
FROM IDENTIFIER(my_catalog || '.silver.orders_enriched')
LIMIT 20;

order_id,customer_id,product_id,order_date,quantity,unit_price,line_total,discount_pct,shipping_cost,order_status,payment_method,customer_name,customer_segment,customer_city,customer_state,product_name,product_category,product_brand,product_list_price
14251,612,16,2025-02-28,2,408.64,817.28,0.0754,20.57,completed,debit_card,Douglas Young,corporate,Houston,TX,VoltStream Ultra Cable,electronics,VoltStream,179.38
14252,702,28,2024-07-27,3,213.67,641.01,0.0732,24.83,cancelled,paypal,Samantha Young,consumer,Detroit,MI,NovaByte Advanced Speaker,electronics,NovaByte,256.16
14253,2008,195,2025-11-03,1,453.77,453.77,0.0132,5.36,completed,paypal,Diane Hill,corporate,Sacramento,CA,SilkEssence Revive Skincare,beauty,SilkEssence,363.25
14254,1693,105,2025-04-05,2,272.30,544.60,0.1434,12.76,completed,credit_card,Helen Allen,consumer,Richmond,VA,CoreMotion Flex Dumbbell,sports,CoreMotion,381.20
14255,1241,161,2025-09-16,1,49.85,49.85,0.2471,17.33,completed,apple_pay,Rachel Stewart,home_office,Richmond,VA,FreshField Gourmet Supplement,food,FreshField,120.55
14256,2167,182,2024-11-20,4,26.72,106.88,0.1108,24.53,completed,debit_card,Christopher Perez,corporate,Kansas City,MO,PureAura Crystal Haircare,beauty,PureAura,428.97
14257,2143,190,2024-11-24,8,47.39,379.12,0.1555,22.43,completed,apple_pay,Joseph Castillo,corporate,Detroit,MI,BloomBeauty Silk Sun Care,beauty,BloomBeauty,108.60
14258,1705,177,2025-12-21,3,453.24,1359.72,0.3,3.93,completed,credit_card,Melissa Myers,consumer,Kansas City,MO,AquaBliss Hydra Sun Care,beauty,AquaBliss,73.52
14259,1973,154,2024-05-17,7,371.86,2603.02,0.1362,22.47,completed,credit_card,Anthony Clark,consumer,Boulder,CO,OceanCatch Organic Cereal,food,OceanCatch,430.08
14260,1627,78,2025-08-21,1,485.68,485.68,0.1525,9.81,completed,debit_card,Charles Adams,consumer,Miami,FL,HarborHouse Artisan Storage,home,HarborHouse,60.93


In [0]:
SELECT COUNT(*) AS row_count
FROM IDENTIFIER(my_catalog || '.silver.orders_enriched');

row_count
15000


<div style="border-left: 4px solid #2e7d32; background: #f7fdf4; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#1b5e20; margin-bottom:6px; font-size: 1.1em;">What to observe</strong>
  <div style="color:#333;">
    <ul>
      <li>Each order now includes the <strong>customer name</strong>, <strong>segment</strong>, and <strong>location</strong> from the customers table</li>
      <li>Each order includes the <strong>product name</strong>, <strong>category</strong>, and <strong>brand</strong> from the products table</li>
      <li>The <code>line_total</code> column was computed as <code>quantity * unit_price</code></li>
      <li><code>COALESCE</code> replaced any null customer names with <code>'Unknown'</code> and null product names with <code>'Unknown Product'</code></li>
      <li>Dates and numeric fields are now properly typed</li>
    </ul>
    <p style="margin-top:10px;"><strong>What to observe:</strong> Materialized Views store the precomputed result. This query would be expensive to run repeatedly against raw data, but the MV makes it instant for downstream consumers.</p>
  </div>
</div>

## F. Incremental Processing: Manual Refresh

Now we test the incremental pipeline. We will:
1. Capture baseline counts
2. Add a new batch of order files to the Volume
3. Manually refresh the Streaming Table
4. Manually refresh the Materialized View
5. Verify that only the new rows were processed

This demonstrates incremental processing end-to-end. In Section G, we will automate the MV refresh with `TRIGGER ON UPDATE`.

### F1. Capture Baseline Counts

Before adding new data, record the current row counts for both objects.

In [0]:
SELECT 'bronze.raw_orders (ST)' AS table_name, COUNT(*) AS row_count
FROM IDENTIFIER(my_catalog || '.bronze.raw_orders')
UNION ALL
SELECT 'bronze.raw_products (ST)', COUNT(*)
FROM IDENTIFIER(my_catalog || '.bronze.raw_products')
UNION ALL
SELECT 'silver.orders_enriched (MV)', COUNT(*)
FROM IDENTIFIER(my_catalog || '.silver.orders_enriched');

table_name,row_count
bronze.raw_orders (ST),15000
bronze.raw_products (ST),200
silver.orders_enriched (MV),15000


### F2. Add New Order Files to the Volume

In a real pipeline, new data arrives as files landing in cloud storage. We simulate this by writing a new CSV file into the same Volume directory that our Streaming Table monitors. The ST does not know about this file yet. It will only pick it up when we refresh.

In [0]:
EXECUTE IMMEDIATE
  CONCAT(
    'INSERT OVERWRITE DIRECTORY \'/Volumes/' || my_catalog || '/data/course_data/raw_orders/raw_orders_new_batch\'
     USING CSV OPTIONS (\'header\' = \'true\')
     SELECT * FROM ' || my_catalog || '.data.raw_orders_new_batch'
  );

<div style="border-left: 4px solid #2e7d32; background: #f7fdf4; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#1b5e20; margin-bottom:6px; font-size: 1.1em;">What to observe</strong>
  <div style="color:#333;">
    A new CSV file containing approximately <strong>2,000 new order records</strong> has been written to the Volume. The Streaming Table has not yet ingested it.
  </div>
</div>

### F3. Refresh the Pipeline: ST then MV

We manually refresh both objects in sequence: the Streaming Table first (Auto Loader ingests the new file), then the Materialized View (which processes only the changed rows via `REFRESH POLICY INCREMENTAL`). Without a trigger configured, both refreshes must be run explicitly.

In [0]:
REFRESH STREAMING TABLE IDENTIFIER(my_catalog || '.bronze.raw_orders');

REFRESH MATERIALIZED VIEW IDENTIFIER(my_catalog || '.silver.orders_enriched');

message
The operation was successfully executed.


result
The operation was successfully executed.


### F4. Explore the Auto-Created Pipelines (While Refreshes Run)

While the refreshes execute, let's explore the auto-created pipelines backing our objects.

<div style="border-left: 4px solid #d97706; background: #fffbeb; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#92400e; margin-bottom:6px; font-size: 1.1em;">Explore in the UI</strong>
  <div style="color:#333;">
    <ol>
      <li>In the left sidebar, click <strong>Jobs and Pipelines</strong></li>
      <li>Click the <strong>Pipelines</strong> tab</li>
      <li>Find the pipeline named <strong>ST-learn_etl_pipelines_sql_YOUR_USERNAME.bronze.raw_orders</strong></li>
      <li>Click the <strong>Performance</strong> tab and look at the most recent refresh entry</li>
      <li>Check <strong>Rows Read / Rows Written</strong>: this confirms only ~2,000 rows were processed in the last refresh</li>
      <li>Go back and find <strong>MV-learn_etl_pipelines_sql_YOUR_USERNAME.silver.orders_enriched</strong></li>
      <li>Check the <strong>Incrementalization</strong> column on the <strong>Tables</strong> tab: it should show <strong>Incremental</strong>. You can also see this label below the task node on the pipeline DAG view.</li>
    </ol>
    <p style="margin-top:8px;">The Pipelines UI gives you full visibility into what happened under the hood: Rows Read/Written per refresh, query profiles, and incrementalization status.</p>
  </div>
</div>

<div style="border-left: 4px solid #2e7d32; background: #f7fdf4; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#1b5e20; margin-bottom:6px; font-size: 1.1em;">What to observe</strong>
  <div style="color:#333;">
    <ul>
      <li>The ST refresh ingested only the new CSV file. The ~15,000 rows from before were <strong>not</strong> reprocessed.</li>
      <li>The MV refresh processed only the ~2,000 new rows from the ST. Not the full dataset. This is <code>REFRESH POLICY INCREMENTAL</code> in action.</li>
    </ul>
  </div>
</div>

### F5. Verify the Incremental Refresh

In [0]:
SELECT 'bronze.raw_orders (ST)' AS table_name, COUNT(*) AS row_count
FROM IDENTIFIER(my_catalog || '.bronze.raw_orders')
UNION ALL
SELECT 'bronze.raw_products (ST)', COUNT(*)
FROM IDENTIFIER(my_catalog || '.bronze.raw_products')
UNION ALL
SELECT 'silver.orders_enriched (MV)', COUNT(*)
FROM IDENTIFIER(my_catalog || '.silver.orders_enriched');

table_name,row_count
bronze.raw_orders (ST),17000
bronze.raw_products (ST),200
silver.orders_enriched (MV),17000


<div style="border-left: 4px solid #2e7d32; background: #f7fdf4; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#1b5e20; margin-bottom:6px; font-size: 1.1em;">What to observe</strong>
  <div style="color:#333;">
    <ul>
      <li>Both the ST and MV row counts increased by approximately <strong>2,000 rows</strong></li>
      <li>The totals should now be approximately <strong>17,000 rows</strong> each</li>
      <li>The incremental pipeline works: new data flows from CSV → Bronze ST → Silver MV, processing only the delta</li>
    </ul>
  </div>
</div>

### F6. Examine the Refresh History

In [0]:
EXECUTE IMMEDIATE
  CONCAT('DESCRIBE HISTORY ' || my_catalog || '.bronze.raw_orders LIMIT 5');

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-11-09T03:32:41.000Z,74546832009708,joemar.brillo@outlook.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 6ec9b6b7-cea8-48e9-8ac6-2f6cd044c72f, epochId -> 1, statsOnLoad -> true)",null,null,f2697583-0010-47fb-9c4b-0ed9bfd81f05,1109-033201-k900wsoj-v2n,2,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 2000, numOutputBytes -> 62170, numAddedFiles -> 1)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-09T03:25:52.000Z,74546832009708,joemar.brillo@outlook.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 6ec9b6b7-cea8-48e9-8ac6-2f6cd044c72f, epochId -> 0, statsOnLoad -> true)",null,null,b8852326-f5c9-4941-b1f2-abb2178fd7c9,1109-032518-5n6zyq2i-v2n,1,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 15000, numOutputBytes -> 416863, numAddedFiles -> 1)",null,Databricks-Runtime/dlt:19.13.4-delta-pipelines-aarch64-photon-dlt-release-dp-20261030-rc0-commit-2402da1-image-d382424
1,2026-11-09T03:25:42.000Z,74546832009708,joemar.brillo@outlook.com,DLT SETUP,"Map(pipelineId -> 9e3b8f6c-5f7a-4056-b453-39a3a1ef2fe8, updateId -> 8914f9d3-00b3-43a8-99db-f922f0ae2d11)",null,null,null,1109-032518-5n6zyq2i-v2n,0,WriteSerializable,false,Map(),null,Databricks-Runtime/dlt:19.13.4-delta-pipelines-aarch64-photon-dlt-release-dp-20261030-rc0-commit-2402da1-image-d382424
0,2026-11-09T03:25:41.000Z,74546832009708,joemar.brillo@outlook.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""spark.sql.internal.pipelines.parentTableId"":""d8737009-31fb-4f6e-a18c-45b82ba537fb"",""spark.sql.internal.unityCatalog.internalEntity.isListable"":""false"",""spark.sql.internal.unityCatalog.internalEntity.inheritsPolicy"":""false"",""delta.enableRowTracking"":""true"",""io.unitycatalog.tableId"":""39e8f45f-d658-44f2-93ee-697b07d3bf36"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-bd174682-e843-4d3d-b740-b09b55177be6"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-fde920cf-1bb9-4c53-8ae6-001946ff4c81"",""spark.internal.pipelines.table_artifact_type"":""st_backing_table"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,null,null,1109-032518-5n6zyq2i-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/dlt:19.13.4-delta-pipelines-aarch64-photon-dlt-release-dp-20261030-rc0-commit-2402da1-image-d382424


<div style="border-left: 4px solid #2e7d32; background: #f7fdf4; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#1b5e20; margin-bottom:6px; font-size: 1.1em;">What to observe</strong>
  <div style="color:#333;">
    <p>Look at the <code>operationMetrics</code> column:</p>
    <ul>
      <li>The most recent version shows ~2,000 rows written (the new batch)</li>
      <li>Earlier versions show the initial load of ~15,000 rows</li>
      <li>Auto Loader remembered which files were already processed. Only the new file was ingested.</li>
    </ul>
  </div>
</div>

## G. Automate with TRIGGER ON UPDATE

The manual workflow works. But in production you don't want to remember to refresh every downstream object. **TRIGGER ON UPDATE** automates this: when the ST is refreshed, the MV refreshes automatically.

We will:
1. Add TRIGGER ON UPDATE to the MV
2. Add one more file to the Volume
3. Refresh only the ST. Watch the MV trigger fire automatically.

### G1. Set TRIGGER ON UPDATE

This `ALTER` adds the trigger directly. No need to go to the SQL Editor for this one.

In [0]:
ALTER MATERIALIZED VIEW silver.orders_enriched
ADD TRIGGER ON UPDATE;

<div style="border-left: 4px solid #2e7d32; background: #f7fdf4; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#1b5e20; margin-bottom:6px; font-size: 1.1em;">What to observe</strong>
  <div style="color:#333;">
    <p>We have now configured a two-part refresh strategy:</p>
    <ul>
      <li><strong>REFRESH POLICY INCREMENTAL</strong>: the MV processes only changed rows (the <em>how</em>)</li>
      <li><strong>TRIGGER ON UPDATE</strong>: the MV refreshes automatically when the ST is refreshed (the <em>when</em>)</li>
    </ul>
    <p style="margin-top:10px;">Together, these two settings mean we only need to refresh the Streaming Table. The MV takes care of itself.</p>
  </div>
</div>

### How TRIGGER ON UPDATE Works

<div style="border-left: 4px solid #1565c0; background: #f0f7ff; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <div style="color:#333;">
    <p>When you add <code>TRIGGER ON UPDATE</code> to a Materialized View, the platform registers a <strong>dependency link</strong> between the MV and its source Streaming Tables. Here is what happens at runtime:</p>
    <ol>
      <li>You (or a schedule) run <code>REFRESH STREAMING TABLE</code></li>
      <li>The ST pipeline completes and writes new data</li>
      <li>The platform detects the upstream change and <strong>automatically enqueues</strong> a refresh of the MV</li>
      <li>The MV pipeline starts on serverless compute and processes only the changed rows</li>
    </ol>
    <p style="margin-top:10px;"><strong>Verify the trigger is set:</strong></p>
    <ol>
      <li>In the left sidebar, click <strong>Catalog</strong></li>
      <li>Navigate to your catalog &rarr; <code>silver</code> schema &rarr; click <strong>orders_enriched</strong></li>
      <li>On the <strong>Overview</strong> tab, find <strong>See refresh schedule</strong></li>
      <li>It should now show <strong>Trigger on update</strong>, confirming the trigger is configured</li>
    </ol>
  </div>
</div>

### G2. Add One More File and Trigger the Cascade

Let's add one more small batch of data and refresh the ST. This time, we will **not** manually refresh the MV. The trigger will do it for us. We just need to confirm the trigger fires.

In [0]:
EXECUTE IMMEDIATE
  CONCAT(
    'INSERT OVERWRITE DIRECTORY \'/Volumes/' || my_catalog || '/data/course_data/raw_orders/raw_orders_trigger_test\'
     USING CSV OPTIONS (\'header\' = \'true\')
     SELECT * FROM ' || my_catalog || '.data.raw_orders_new_batch LIMIT 500'
  );

### G3. Refresh the ST: Watch the Trigger Fire

In [0]:
REFRESH STREAMING TABLE IDENTIFIER(my_catalog || '.bronze.raw_orders');

message
The operation was successfully executed.


<div style="border-left: 4px solid #2e7d32; background: #f7fdf4; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#1b5e20; margin-bottom:6px; font-size: 1.1em;">What to observe</strong>
  <div style="color:#333;">
    The ST refresh completed and ingested the new file. Now confirm the MV trigger fired:
    <ol style="margin-top:8px;">
      <li>In the left sidebar, click <strong>Catalog</strong></li>
      <li>Navigate to your catalog &rarr; <code>silver</code> schema &rarr; click <strong>orders_enriched</strong></li>
      <li>On the <strong>Overview</strong> tab, check <strong>Current refresh status</strong>: you should see <strong>Waiting for resources...</strong></li>
      <li>Click <strong>See refresh details</strong> to open the pipeline and watch it allocate compute</li>
    </ol>
    <p style="margin-top:10px;">That's TRIGGER ON UPDATE in action. We did <strong>not</strong> run <code>REFRESH MATERIALIZED VIEW</code>. The engine detected the upstream ST change and automatically started the MV refresh. You don't need to wait for it to complete. The key takeaway is that the trigger <strong>fired automatically</strong>.</p>
  </div>
</div>

<div style="border-left: 4px solid #1565c0; background: #f0f7ff; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#1565c0; margin-bottom:6px; font-size: 1.1em;">Note</strong>
  <div style="color:#333;">
    You can also check the pipeline directly via <strong>Jobs and Pipelines</strong> &rarr; <strong>Pipelines</strong> tab &rarr; open <strong>MV-learn_etl_pipelines_sql_YOUR_USERNAME.silver.orders_enriched</strong>. The pipeline list page may not show activity yet. Open the specific pipeline to see the <strong>Waiting for resources</strong> status.
  </div>
  </div>
</div>

## H. Key Summary and Takeaways

In this demo, we built a complete Bronze-to-Silver incremental pipeline. Here are the key patterns:

| Concept | What We Did | Why It Matters |
|---------|-------------|----------------|
| **EXPLAIN** | Ran EXPLAIN before and after setting properties | Pre-deployment diagnostic: tells you exactly what to fix for incremental refresh |
| **Streaming Tables** | Created `bronze.raw_orders` and `bronze.raw_products` with Auto Loader | Append-only ingestion with automatic file tracking: only new files are processed |
| **Materialized View** | Created `silver.orders_enriched` with joins | Precomputed, analysis-ready data: complex query runs once, not on every read |
| **REFRESH POLICY INCREMENTAL** | Set on the MV | Processes only changed rows, not the full dataset (the *how*) |
| **Manual refresh** | Refreshed ST then MV explicitly | Proved incremental processing works end-to-end |
| **TRIGGER ON UPDATE** | Connected MV to ST | MV auto-refreshes when ST is refreshed. No manual step needed (the *when*). |

**To summarize:** `REFRESH POLICY INCREMENTAL` controls *how* the MV refreshes (only changed rows). `TRIGGER ON UPDATE` controls *when* (automatically, after the ST). Together, they build pipelines where refreshing the first object in the chain is the *only* step. Everything downstream handles itself.

&copy; 2026 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the <a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/>
<a href="https://databricks.com/privacy-policy">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use">Terms of Use</a> | <a href="https://help.databricks.com/">Support</a>